In [50]:
import os

print(os.listdir())

['.config', 'cleaned_data.csv', '.ipynb_checkpoints', 'heart_disease_dataset (2).csv', 'plots', 'sample_data']


In [51]:
import pandas as pd

print("=" * 70)
print("TASK 1: LOAD THE DATASET AND LOOK AT IT")
print("=" * 70)

# STEP 1: Load the CSV file into a pandas DataFrame

df = pd.read_csv('/content/heart_disease_dataset (2).csv')

print("\n--- First 5 rows of the data ---")
print(df.head())

# (int64 = whole numbers, float64 = decimal numbers, object = text/strings)
print("\n--- Data types of each column ---")
print(df.dtypes)

print("\n--- Shape of the DataFrame (rows, columns) ---")
print(df.shape)


print("\n" + "=" * 70)
print("TASK 2: FIND AND HANDLE MISSING VALUES (NULLS)")
print("=" * 70)

# STEP 2a: Count how many missing (null) values are in each column

null_counts = df.isnull().sum()
print("\n--- Number of missing values per column ---")
print(null_counts)


# STEP 2b: Turn those counts into percentages

null_percent = (df.isnull().sum() / df.shape[0]) * 100
print("\n--- Percentage of missing values per column ---")
print(null_percent.round(2))

# STEP 2c: Report which columns have MORE than 20% missing values

high_null_cols = null_percent[null_percent > 20].index.tolist()
print("\n--- Columns with more than 20% missing values (NOT filled in) ---")
print(high_null_cols)

# STEP 2d: For columns with LESS than 20% missing values,

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("\n--- Filling missing values for numeric columns with LESS than 20% nulls ---")
for col in numeric_cols:

    if col in high_null_cols:
        print(f" Skipping '{col}")
        continue

    if null_counts[col] == 0:
        continue
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"  Filled '{col}' missing values with median = {median_value}")

print("\n--- Missing value counts AFTER filling ---")
print(df.isnull().sum())


print("\n" + "=" * 70)
print("TASK 3: FIND AND REMOVE DUPLICATE ROWS")
print("=" * 70)


# STEP 3a: Count how many duplicate rows exist


duplicate_count = df.duplicated().sum()
print(f"\nNumber of duplicate rows found: {duplicate_count}")

null_percent_before = (df.isnull().sum() / df.shape[0]) * 100

# STEP 3b: Remove the duplicate rows

rows_before = df.shape[0]
df = df.drop_duplicates()
rows_after = df.shape[0]

print(f"Rows before removing duplicates: {rows_before}")
print(f"Rows after removing duplicates:  {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")


# STEP 3c: Check if removing duplicates changed the null percentages

null_percent_after = (df.isnull().sum() / df.shape[0]) * 100

comparison = pd.DataFrame({
    'Null % Before': null_percent_before.round(2),
    'Null % After': null_percent_after.round(2)
})
print(comparison)


print("\n" + "=" * 70)
print("TASK 4: FIX INCORRECT DATA TYPES")
print("=" * 70)


# STEP 4a: Measure memory usage BEFORE fixing data types

memory_before = df.memory_usage(deep=True).sum()
print(f"\nTotal memory usage BEFORE cleanup: {memory_before:,} bytes "
      f"({memory_before / 1024:.2f} KB)")

# STEP 4b: Fix the 'Age' column


print(df[pd.to_numeric(df['Age'], errors='coerce').isna()]['Age'].unique())

df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

if df['Age'].isnull().sum() > 0:
    age_median = df['Age'].median()
    df['Age'] = df['Age'].fillna(age_median)
    print(f"Filled the new 'Age' missing values with the median: {age_median}")

# STEP 4c: Convert repetitive text columns to the 'category' dtype

columns_to_convert = ['Sex', 'Smoker', 'BloodType', 'FamilyHistory', 'Diabetes', 'HeartDisease']

print("\n--- Converting repetitive text columns to 'category' dtype ---")
for col in columns_to_convert:
    df[col] = df[col].astype('category')
    print(f"  '{col}' converted. Unique values: {df[col].cat.categories.tolist()}")

# STEP 4d: Measure memory usage AFTER fixing data types

memory_after = df.memory_usage(deep=True).sum()
print(f"\nTotal memory usage AFTER cleanup: {memory_after:,} bytes "
      f"({memory_after / 1024:.2f} KB)")

memory_saved = memory_before - memory_after
percent_saved = (memory_saved / memory_before) * 100
print(f"Memory saved: {memory_saved:,} bytes ({percent_saved:.1f}% reduction)")




TASK 1: LOAD THE DATASET AND LOOK AT IT

--- First 5 rows of the data ---
  PatientID Age     Sex   BMI Smoker BloodType  SystolicBP  DiastolicBP  \
0   PT00501  67  Female  22.7     No        B+         144           73   
1   PT00200  84  Female  16.3     No        B+         139           71   
2   PT00613  69    Male  33.3    Yes        O-         107           70   
3   PT00440  68  Female  18.0     No        O+          99           70   
4   PT00108  22  Female  29.3     No        A+         117           73   

   Cholesterol  GlucoseMgDl  RestingHR  ExerciseHrsPerWeek FamilyHistory  \
0        181.0          NaN         68                 0.3            No   
1        223.0        113.0         57                 0.5            No   
2        152.0        149.0         66                 0.8            No   
3        192.0         67.0         70                 0.2            No   
4        179.0          NaN         49                10.2            No   

  Diabetes  Diagno

In [52]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # lets matplotlib save images without needing a screen
import matplotlib.pyplot as plt
import seaborn as sns

# Make a folder to store our chart images
os.makedirs('plots', exist_ok=True)

print("\n" + "=" * 70)
print("TASK 5: DESCRIPTIVE STATISTICS AND SKEWNESS")
print("=" * 70)

# STEP 5a: df.describe() gives us count, mean, std, min, 25%/50%/75%, max

print("\n--- Descriptive statistics for all numeric columns ---")
print(df[numeric_cols].describe())

skewness = df[numeric_cols].skew()
print("\n--- Skewness of each numeric column (sorted by strength) ---")
print(skewness.reindex(skewness.abs().sort_values(ascending=False).index))

most_skewed_col = skewness.abs().idxmax()
print(f"\nColumn with the highest absolute skewness: '{most_skewed_col}' "
      f"(skew = {skewness[most_skewed_col]:.3f})")


print("\n" + "=" * 70)
print("TASK 6: OUTLIER DETECTION WITH IQR")
print("=" * 70)

outlier_summary = {}
for col in ['ExerciseHrsPerWeek', 'Cholesterol']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_summary[col] = len(outliers)

    print(f"\n--- {col} ---")
    print(f"  Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
    print(f"  Valid range: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Number of outlier rows: {len(outliers)}")

# NOTE: We are NOT dropping these outliers. See the README for the
# reasoning.

print("\n" + "=" * 70)
print("TASK 7: VISUALIZATIONS")
print("=" * 70)
sns.set_style('whitegrid')

# 1. LINE PLOT

plt.figure(figsize=(10, 5))
plt.plot(df.index, df['Cholesterol'], linewidth=0.8, color='#c0392b')
plt.title('Cholesterol Level by Row Index')
plt.xlabel('Row Index (each row = one patient)')
plt.ylabel('Cholesterol (mg/dL)')
plt.tight_layout()
plt.savefig('plots/1_line_plot_cholesterol.png', dpi=120)
plt.close()
print("Saved: plots/1_line_plot_cholesterol.png")

# 2. BAR CHART

mean_by_bloodtype = df.groupby('BloodType', observed=True)['Cholesterol'].mean().sort_values()
plt.figure(figsize=(10, 5))
mean_by_bloodtype.plot.bar(color='#2980b9')
plt.title('Average Cholesterol by Blood Type')
plt.xlabel('Blood Type')
plt.ylabel('Average Cholesterol (mg/dL)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('plots/2_bar_chart_cholesterol_by_bloodtype.png', dpi=120)
plt.close()
print("Saved: plots/2_bar_chart_cholesterol_by_bloodtype.png")


# 3. HISTOGRAM

plt.figure(figsize=(10, 5))
sns.histplot(df[most_skewed_col].dropna(), bins=20, color='#8e44ad')
plt.title(f'Distribution of {most_skewed_col} (Most Skewed Column)')
plt.xlabel(most_skewed_col)
plt.ylabel('Number of Patients')
plt.tight_layout()
plt.savefig('plots/3_histogram_most_skewed.png', dpi=120)
plt.close()
print("Saved: plots/3_histogram_most_skewed.png")

# 4. SCATTER PLOT - two numeric columns we'd expect to be related

# Systolic and diastolic blood pressure are the "top" and "bottom" numbers
# in a blood pressure reading -- clinically, they usually move together.
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='SystolicBP', y='DiastolicBP', alpha=0.5, color='#16a085')
plt.title('Systolic vs Diastolic Blood Pressure')
plt.xlabel('Systolic BP (mmHg)')
plt.ylabel('Diastolic BP (mmHg)')
plt.tight_layout()
plt.savefig('plots/4_scatter_systolic_vs_diastolic.png', dpi=120)
plt.close()
print("Saved: plots/4_scatter_systolic_vs_diastolic.png")

bp_corr = df['SystolicBP'].corr(df['DiastolicBP'])
print(f"Correlation between SystolicBP and DiastolicBP: {bp_corr:.3f}")


# 5. BOX PLOT - a numeric column split by a categorical column

plt.figure(figsize=(7, 6))
sns.boxplot(data=df, x='HeartDisease', y='Cholesterol', palette='Set2', hue='HeartDisease', legend=False)
plt.title('Cholesterol Distribution by Heart Disease Status')
plt.xlabel('Heart Disease')
plt.ylabel('Cholesterol (mg/dL)')
plt.tight_layout()
plt.savefig('plots/5_boxplot_cholesterol_by_heartdisease.png', dpi=120)
plt.close()
print("Saved: plots/5_boxplot_cholesterol_by_heartdisease.png")

print("\n" + "=" * 70)
print("TASK 8: CORRELATION HEAT MAP")
print("=" * 70)


corr_matrix = df[numeric_cols].corr()  # Pearson correlation by default

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heat Map (Pearson)')
plt.tight_layout()
plt.savefig('plots/6_correlation_heatmap.png', dpi=120)
plt.close()
print("Saved: plots/6_correlation_heatmap.png")

# Find the pair of DIFFERENT columns with the highest absolute correlation
corr_abs_vals = corr_matrix.abs().values.copy()
np.fill_diagonal(corr_abs_vals, 0)  # ignore each column's correlation with itself (always 1.0)
corr_abs = pd.DataFrame(corr_abs_vals, index=corr_matrix.index, columns=corr_matrix.columns)
strongest_pair = corr_abs.unstack().sort_values(ascending=False).index[0]
strongest_value = corr_matrix.loc[strongest_pair[0], strongest_pair[1]]
print(f"\nStrongest correlation pair: {strongest_pair[0]} & {strongest_pair[1]} "
      f"(r = {strongest_value:.3f})")





TASK 5: DESCRIPTIVE STATISTICS AND SKEWNESS

--- Descriptive statistics for all numeric columns ---
              BMI  SystolicBP  DiastolicBP  Cholesterol  GlucoseMgDl  \
count  650.000000  650.000000   650.000000   650.000000   494.000000   
mean    27.003231  125.066154    79.875385   200.952308    99.945344   
std      5.309326   16.170707    10.310952    34.929635    22.928212   
min     15.000000   85.000000    50.000000   110.000000    60.000000   
25%     23.500000  114.000000    73.000000   181.000000    84.000000   
50%     27.000000  125.000000    80.000000   201.000000    99.000000   
75%     30.300000  136.750000    87.000000   224.000000   116.000000   
max     43.100000  171.000000   105.000000   317.000000   174.000000   

        RestingHR  ExerciseHrsPerWeek  DiagnosisYear  
count  650.000000          650.000000      97.000000  
mean    71.993846            2.204462    2014.484536  
std     11.342135            2.135992       5.870323  
min     45.000000            0

In [53]:
print("\n" + "=" * 70)
print("TASK 9a: IMPUTATION STRATEGY COMPARISON")
print("=" * 70)

# The two columns with the highest absolute skewness (from Task 5)
top_2_skewed = skewness.abs().sort_values(ascending=False).index[:2].tolist()
print(f"\nTwo most skewed columns: {top_2_skewed}")

print("\n--- Mean vs Median BEFORE imputation ---")
for col in top_2_skewed:
    col_mean = df[col].mean()
    col_median = df[col].median()
    print(f"  {col}: mean = {col_mean:.2f}, median = {col_median:.2f}, "
          f"skew = {skewness[col]:.2f}, nulls before fill = {df[col].isnull().sum()}")
print("\nChosen strategy for both columns: MEDIAN (they are skewed, "
      "so the mean is distorted by extreme values in the tail).")

for col in top_2_skewed:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

print("\n--- Confirming no nulls remain in these two columns ---")
print(df[top_2_skewed].isnull().sum())

null_percent = (df.isnull().sum() / df.shape[0]) * 100
high_null_cols = null_percent[null_percent > 20].index.tolist()
print(f"\nColumns still above 20% missing (left unfilled): {high_null_cols}")

for col in numeric_cols:
    if col in top_2_skewed or col in high_null_cols:
        continue
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f"Filled remaining nulls in '{col}' with the median.")


print("\n" + "=" * 70)
print("TASK 9b: SPEARMAN RANK CORRELATION")
print("=" * 70)

# Pearson correlation and Spearman correlation

spearman_matrix = df[numeric_cols].corr(method='spearman')
pearson_matrix = df[numeric_cols].corr(method='pearson')

print("\n--- Pearson correlation matrix ---")
print(pearson_matrix.round(3))
print("\n--- Spearman correlation matrix ---")
print(spearman_matrix.round(3))

# Compute the absolute difference between the two matrices
diff_vals = (spearman_matrix - pearson_matrix).abs().values.copy()
np.fill_diagonal(diff_vals, 0)  # ignore the diagonal (always 0 anyway)
diff_matrix = pd.DataFrame(diff_vals, index=spearman_matrix.index, columns=spearman_matrix.columns)

# Find the top 3 column pairs with the biggest Spearman-vs-Pearson difference
diff_pairs = diff_matrix.unstack().sort_values(ascending=False)
seen_pairs = set()
top_3_diff = []
for (col_a, col_b), diff_value in diff_pairs.items():
    if col_a == col_b:
        continue
    pair_key = tuple(sorted([col_a, col_b]))
    if pair_key in seen_pairs:
        continue
    seen_pairs.add(pair_key)
    top_3_diff.append((col_a, col_b, diff_value,
                        pearson_matrix.loc[col_a, col_b],
                        spearman_matrix.loc[col_a, col_b]))
    if len(top_3_diff) == 3:
        break

print("\n--- Top 3 pairs where Spearman and Pearson disagree the most ---")
diff_table = pd.DataFrame(top_3_diff, columns=['Column A', 'Column B',
                                                '|Spearman - Pearson|',
                                                'Pearson r', 'Spearman r'])
print(diff_table.round(3))


print("\n" + "=" * 70)
print("TASK 9c: GROUPED AGGREGATION")
print("=" * 70)

# We group by HeartDisease (categorical) and look at Cholesterol (numeric)
group_stats = df.groupby('HeartDisease', observed=True)['Cholesterol'].agg(['mean', 'std', 'count'])
print("\n--- Cholesterol statistics grouped by Heart Disease status ---")
print(group_stats.round(2))

highest_mean_group = group_stats['mean'].idxmax()
highest_std_group = group_stats['std'].idxmax()
mean_ratio = group_stats['mean'].max() / group_stats['mean'].min()

print(f"\nGroup with highest mean Cholesterol: '{highest_mean_group}' "
      f"({group_stats['mean'].max():.2f})")
print(f"Group with highest std (spread): '{highest_std_group}' "
      f"({group_stats['std'].max():.2f})")
print(f"Ratio of highest group mean to lowest group mean: {mean_ratio:.3f}")


print("\n" + "=" * 70)
print(" PART 10 SAVE FINAL CLEANED DATASET")
print("=" * 70)

df.to_csv('cleaned_data.csv', index=False)
print("\nSaved as 'cleaned_data.csv'")
print("Final shape:", df.shape)
print("Final null counts:\n", df.isnull().sum())




TASK 9a: IMPUTATION STRATEGY COMPARISON

Two most skewed columns: ['ExerciseHrsPerWeek', 'GlucoseMgDl']

--- Mean vs Median BEFORE imputation ---
  ExerciseHrsPerWeek: mean = 2.20, median = 1.60, skew = 2.18, nulls before fill = 0
  GlucoseMgDl: mean = 99.95, median = 99.00, skew = 0.26, nulls before fill = 156

Chosen strategy for both columns: MEDIAN (they are skewed, so the mean is distorted by extreme values in the tail).

--- Confirming no nulls remain in these two columns ---
ExerciseHrsPerWeek    0
GlucoseMgDl           0
dtype: int64

Columns still above 20% missing (left unfilled): ['DiagnosisYear']

TASK 9b: SPEARMAN RANK CORRELATION

--- Pearson correlation matrix ---
                      BMI  SystolicBP  DiastolicBP  Cholesterol  GlucoseMgDl  \
BMI                 1.000      -0.029        0.017       -0.026        0.020   
SystolicBP         -0.029       1.000        0.038       -0.039       -0.013   
DiastolicBP         0.017       0.038        1.000       -0.049       -